In [11]:
# Install imageio[ffmpeg] into the kernel environment and verify ffmpeg availability
# This cell uses the notebook %pip magic to ensure installation into the running kernel.
# 실행: 이 셀을 실행하면 pip가 업그레이드되고 imageio[ffmpeg]가 설치됩니다.

# NOTE: if your environment uses conda, %pip will still install into the kernel's Python.

%pip install --upgrade pip
%pip install "imageio[ffmpeg]"

# Quick verification
import importlib, shutil
try:
    import imageio
    print('imageio', imageio.__version__)
except Exception as e:
    print('imageio import error:', e)
try:
    import imageio_ffmpeg
    print('imageio_ffmpeg module found')
except Exception as e:
    print('imageio_ffmpeg import error:', e)
print('ffmpeg binary on PATH:', shutil.which('ffmpeg'))


  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 8.6 MB/s  0:00:02 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
imageio 2.33.1
imageio_ffmpeg module found
ffmpeg binary on PATH: /opt/homebrew/bin/ffmpeg


In [7]:
%pip install imageio
%pip install ffmpeg

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py) ...   Downloading ffmpeg-1.4.tar.gz (5.1 kB)
  Preparing metadata (setup.py) ... -done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6082 sha256=5da644ca9345c7abc8ed19d1aee5529ba634f43a41d84f3c6a4e7bf775c41fe7
  Stored in directory: /Users/yiji/Library/Caches/pip/wheels/26/21/0c/c26e09dff860a9071683e279445262346e008a9a1d2142c4ad
Successfully built ffmpeg
done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6082 sha256=5da644ca9345c7abc8ed19d1aee5529ba634f43a41d84f3c6a4e7bf775c41fe7
  Stored in directory: /Users/yiji/Library/Caches/pip/wheels/26/21/0c/c26e09dff860a9071683e279445262346e008a9a1d2142c4ad
Successfully built ffmpeg
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install imageio[ffmpeg]

zsh:1: no matches found: imageio[ffmpeg]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


어느 특정 pedestrian에 대해서 라벨 하나 뽑아서 그에 대한 데이터셋을 만들어보자. 그 사람이 화면 상에 등장하는 동안 함께 등장한 모든 다른 객체를 우선 list up 하고 시작과 끝 프레임을 컬럼으로 해서 기록해보자. 그니까 파일명은 pedestrian_id.csv 이런식으로.

In [2]:
# Create CSV listing all other tracks that co-appear with a target pedestrian
import pandas as pd
from pathlib import Path

def make_coappearance_csv(video_name, target_track_id, base_dir=Path("SDD_datasets"), out_dir=Path("outputs")):
    """
    video_name: string like 'coupa/video0'
    target_track_id: int
    Writes CSV with columns: video, target_track_id, target_start, target_end, other_track_id, other_label, start_frame, end_frame, duration_frames
    Returns path to saved CSV (Path)
    """
    site, vid = video_name.split('/')
    annotations_path = base_dir / "SDD_raw" / site / vid / "annotations.csv"
    labels_path = base_dir / "SDD_labels" / site / f"{vid}_labels.csv"

    if not annotations_path.exists():
        raise FileNotFoundError(f"annotations.csv not found: {annotations_path}")
    if not labels_path.exists():
        raise FileNotFoundError(f"labels csv not found: {labels_path}")

    df = pd.read_csv(annotations_path)
    labels = pd.read_csv(labels_path)

    # Ensure expected columns
    expected = {"track_id","xmin","ymin","xmax","ymax","frame","lost","occluded","generated","label"}
    if not expected.issubset(set(df.columns)):
        raise ValueError(f"annotations.csv missing expected columns. Found: {list(df.columns)}")

    # target frames
    target_frames = df.loc[df['track_id'] == target_track_id, 'frame']
    if target_frames.empty:
        raise ValueError(f"Target track_id {target_track_id} not present in {annotations_path}")

    t_start = int(target_frames.min())
    t_end = int(target_frames.max())

    # subset annotations to the time window where target is present
    df_window = df.loc[(df['frame'] >= t_start) & (df['frame'] <= t_end)].copy()

    # group by track and find overlap start/end
    agg = df_window.groupby('track_id')['frame'].agg(['min','max']).reset_index()
    agg = agg.rename(columns={'min':'start_frame','max':'end_frame'})

    # remove the target itself from other list (but keep info for completeness)
    others = agg.loc[agg['track_id'] != target_track_id].copy()

    # merge labels (try labels csv first, fallback to annotations' label column)
    if 'track_id' in labels.columns and 'label' in labels.columns:
        others = others.merge(labels.rename(columns={'track_id':'label_track_id','label':'label_name'}),
                               left_on='track_id', right_on='label_track_id', how='left')
        others.drop(columns=['label_track_id'], inplace=True)
    else:
        # try to get label from annotations
        lbls = df[['track_id','label']].drop_duplicates(subset=['track_id'])
        others = others.merge(lbls, on='track_id', how='left')
        others = others.rename(columns={'label':'label_name'})

    # add metadata columns
    others['video'] = video_name
    others['target_track_id'] = target_track_id
    others['target_start'] = t_start
    others['target_end'] = t_end
    others['duration_frames'] = others['end_frame'] - others['start_frame'] + 1

    # reorder columns
    cols = ['video','target_track_id','target_start','target_end','track_id','label_name','start_frame','end_frame','duration_frames']
    out_df = others[cols].rename(columns={'track_id':'other_track_id','label_name':'other_label'})

    # ensure output dir
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_file = out_dir / f"pedestrian_{target_track_id}.csv"
    out_df.to_csv(out_file, index=False)

    print(f"Saved co-appearance CSV: {out_file}  (rows: {len(out_df)})")
    return out_file

# Example usage: generate for coupa/video0, pedestrian track 1
# (You can change target_track_id as needed)

if __name__ == '__main__':
    # default example run when executed as script
    try:
        save_path = make_coappearance_csv('coupa/video0', 1)
    except Exception as e:
        print('Error while creating co-appearance CSV:', e)


Saved co-appearance CSV: outputs/pedestrian_1.csv  (rows: 21)


In [ ]:
# Scoring: compute d_ij, v_ij (approach speed), size_j (by label), TTC and combined score
import numpy as np
import pandas as pd
from pathlib import Path

# Label size mapping (user-specified normalized categories in [0,1]):
# - 사람/스케이트보더: 0.0
# - 자전거, 카트: 0.2
# - 자동차: 0.7
# - 버스, 트럭: 1.0
# If a label is not listed, fallback to 0.5
LABEL_SIZE_MAP = {
    'Pedestrian': 0.0,
    'Skater': 0.0,
    'Biker': 0.2,
    'Bicycle': 0.2,
    'Cart': 0.2,
    'Car': 0.7,
    'Bus': 1.0,
    'Truck': 1.0,
}


def compute_track_centers_and_velocity(df):
    """Given annotations dataframe, compute center x/y and per-frame velocity (px/frame) for each track.
    Returns dataframe with columns: track_id, frame, cx, cy, vx, vy, width, height, area
    """
    df = df.copy()
    df['cx'] = (df['xmin'] + df['xmax']) / 2.0
    df['cy'] = (df['ymin'] + df['ymax']) / 2.0
    df['width'] = df['xmax'] - df['xmin']
    df['height'] = df['ymax'] - df['ymin']
    df['area'] = df['width'] * df['height']

    # compute per-track velocities by differencing consecutive frames
    out_rows = []
    for tid, g in df.groupby('track_id'):
        g_sorted = g.sort_values('frame')
        # compute dt and differences
        g_sorted['vx'] = g_sorted['cx'].diff() / g_sorted['frame'].diff()
        g_sorted['vy'] = g_sorted['cy'].diff() / g_sorted['frame'].diff()
        # fillna with 0 for first frame
        g_sorted['vx'] = g_sorted['vx'].fillna(0.0)
        g_sorted['vy'] = g_sorted['vy'].fillna(0.0)
        out_rows.append(g_sorted[['track_id','frame','cx','cy','vx','vy','width','height','area','label']])
    res = pd.concat(out_rows, ignore_index=True)
    return res


def make_scored_csv(video_name, target_track_id, base_dir=Path('SDD_datasets'), out_dir=Path('outputs'), frame_step=5,
                    d_scale=500.0, v_scale=5.0, ttc_scale=30.0):
    """
    Compute per-frame pairwise (target vs other) scoring and save CSV.
    score components (normalized):
      - distance score s_d: 1 / (1 + d/d_scale)
      - approach speed s_v: tanh(v_approach / v_scale)
      - size s_size: label-based normalized value in [0,1] (user mapping)
      - TTC s_ttc: 1 / (1 + ttc/ttc_scale) (if ttc==inf -> 0)
    Final score = mean of four component scores (1:1:1:1)

    frame_step: sample every N frames to reduce output size. Use 1 for all frames.
    """
    site, vid = video_name.split('/')
    annotations_path = base_dir / 'SDD_raw' / site / vid / 'annotations.csv'
    labels_path = base_dir / 'SDD_labels' / site / f"{vid}_labels.csv"

    df = pd.read_csv(annotations_path)
    labels = pd.read_csv(labels_path)

    # compute centers and velocities
    tracks_df = compute_track_centers_and_velocity(df)

    # target presence frames
    t_frames = tracks_df.loc[tracks_df['track_id']==target_track_id, 'frame']
    if t_frames.empty:
        raise ValueError(f"Target {target_track_id} not found in {annotations_path}")
    t_start = int(t_frames.min())
    t_end = int(t_frames.max())

    # build mapping for label->size value
    # start with user mapping; if other labels appear, add fallback 0.5
    label_map = LABEL_SIZE_MAP.copy()
    # add unknowns found in labels
    unique_labels = pd.concat([labels['label'], df['label']]).drop_duplicates().dropna().astype(str)
    for lbl in unique_labels:
        if lbl not in label_map:
            label_map[lbl] = 0.5

    # decide whether mapping already in [0,1]
    all_vals = np.array(list(label_map.values()), dtype=float)
    if np.nanmin(all_vals) >= 0.0 and np.nanmax(all_vals) <= 1.0:
        # mapping already normalized — use directly
        def label_size_norm(lbl):
            return float(label_map.get(lbl, 0.5))
    else:
        # fallback: normalize to [0,1]
        min_s, max_s = all_vals.min(), all_vals.max()
        def label_size_norm(lbl):
            v = float(label_map.get(lbl, 0.5))
            if max_s > min_s:
                return (v - min_s) / (max_s - min_s)
            return 0.5

    # iterate frames in target window
    rows = []
    for frame in range(t_start, t_end+1, frame_step):
        # get target row at this frame
        ti = tracks_df[(tracks_df['track_id']==target_track_id) & (tracks_df['frame']==frame)]
        if ti.empty:
            continue
        ti = ti.iloc[0]
        tx, ty, tvx, tvy, tarea = ti['cx'], ti['cy'], ti['vx'], ti['vy'], ti['area']

        # other tracks present at same frame
        others = tracks_df[(tracks_df['track_id'] != target_track_id) & (tracks_df['frame']==frame)]
        if others.empty:
            continue
        for _, oj in others.iterrows():
            ox, oy, ovx, ovy = oj['cx'], oj['cy'], oj['vx'], oj['vy']
            other_tid = int(oj['track_id'])
            other_label = oj.get('label', None)
            # distance
            dx = ox - tx
            dy = oy - ty
            d = float(np.hypot(dx, dy))

            # relative velocity (v_j - v_i)
            vrel_x = ovx - tvx
            vrel_y = ovy - tvy
            # unit vector from target -> other
            if d > 1e-6:
                ux, uy = dx / d, dy / d
                rel_along = vrel_x * ux + vrel_y * uy
                # approach speed positive when closing (i.e. rel_along < 0)
                v_approach = max(0.0, -rel_along)
            else:
                ux, uy = 0.0, 0.0
                v_approach = 0.0

            # size by label (normalized) + area ratio
            label_val_norm = label_size_norm(str(other_label))
            area_ratio = (oj['area'] / (tarea + 1e-6))

            # TTC: if v_approach > eps compute d / v else inf
            eps = 1e-6
            if v_approach > eps:
                ttc = d / v_approach
            else:
                ttc = np.inf

            # normalized component scores
            s_d = 1.0 / (1.0 + d / d_scale)
            s_v = float(np.tanh(v_approach / v_scale))
            s_size = float(label_val_norm)
            if np.isinf(ttc):
                s_ttc = 0.0
            else:
                s_ttc = 1.0 / (1.0 + ttc / ttc_scale)

            score = float((s_d + s_v + s_size + s_ttc) / 4.0)

            rows.append({
                'video': video_name,
                'frame': int(frame),
                'target_track_id': int(target_track_id),
                'other_track_id': other_tid,
                'target_cx': float(tx), 'target_cy': float(ty),
                'other_cx': float(ox), 'other_cy': float(oy),
                'd_px': d,
                'v_approach_px_per_frame': v_approach,
                'other_label': other_label,
                'label_size_norm': label_val_norm,
                'area_ratio': area_ratio,
                'ttc_frames': (np.nan if np.isinf(ttc) else float(ttc)),
                's_d': s_d, 's_v': s_v, 's_size': s_size, 's_ttc': s_ttc,
                'score': score,
            })

    out_df = pd.DataFrame(rows)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f"pedestrian_{target_track_id}_scored.csv"
    out_df.to_csv(out_file, index=False)
    print(f"Saved scored CSV: {out_file}  (rows: {len(out_df)})")
    return out_file

# quick run example (frame_step=5 to keep file small)
if __name__ == '__main__':
    try:
        outp = make_scored_csv('coupa/video0', 1, frame_step=5)
    except Exception as e:
        print('Error while making scored csv:', e)

Saved scored CSV: outputs/pedestrian_1_scored.csv  (rows: 1747)


In [ ]:
# Overlay directly onto a background video (MOV/MP4) using scored CSV


import imageio
from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors
from pathlib import Path


def _draw_text_with_outline(draw, xy, text, font, fill, outline_color=(0,0,0), outline_width=2):
    """Draw text with a contrasting outline. Uses stroke parameters if available, else falls back to multiple offset draws."""
    try:
        # Pillow >= 8.7 supports stroke_width/stroke_fill
        draw.text(xy, text, font=font, fill=fill, stroke_width=outline_width, stroke_fill=outline_color)
    except TypeError:
        # fallback: draw the outline by drawing the text multiple times offset by 1..outline_width
        x, y = xy
        for dx in range(-outline_width, outline_width+1):
            for dy in range(-outline_width, outline_width+1):
                if dx == 0 and dy == 0:
                    continue
                draw.text((x+dx, y+dy), text, font=font, fill=outline_color)
        draw.text(xy, text, font=font, fill=fill)


def _text_size(draw, text, font):
    """Robust text size helper: tries several Pillow APIs (font.getsize, draw.textbbox) and falls back to a rough estimate."""
    try:
        # Preferred: font.getsize (available in many Pillow versions)
        return font.getsize(text)
    except Exception:
        try:
            # Modern Pillow: draw.textbbox gives precise bbox
            bbox = draw.textbbox((0, 0), text, font=font)
            return (bbox[2] - bbox[0], bbox[3] - bbox[1])
        except Exception:
            # Last resort: conservative fixed-width estimate
            return (len(text) * 8, 12)


def make_overlay_on_video(video_path, target_track_id, scored_csv_path=None, base_dir=Path('SDD_datasets'),
                          out_path=Path('outputs/pedestrian_{id}_overlay_on_video.mov'), fps=None, frame_step=1,
                          marker_scale=40, box_thickness=3, max_frames=None):
    """
    Overlay scored markers onto an actual video file (video_path).
    - video_path: path to input video (mov/mp4)
    - target_track_id: int
    - scored_csv_path: path to scored CSV (defaults to outputs/pedestrian_{id}_scored.csv)
    - out_path: output path for overlay video

    This version pads frames (white) on the right/bottom to make width/height multiples of 16 so ffmpeg
    will not auto-resize. The original image content is preserved at top-left; padding is added to the
    right and bottom as needed.

    It also ensures text is readable by drawing an outline around ID text and placing a white semi-opaque
    background behind the score text.
    """
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    if scored_csv_path is None:
        scored_csv_path = Path('outputs') / f"pedestrian_{target_track_id}_scored.csv"
    scored_csv_path = Path(scored_csv_path)
    if not scored_csv_path.exists():
        raise FileNotFoundError(f"Scored CSV not found: {scored_csv_path}")

    df_scores = pd.read_csv(scored_csv_path)
    annotations_path = base_dir / video_path.parts[-3] / video_path.parts[-2] / 'annotations.csv'  # heuristic: SDD_datasets/SDD_raw/<site>/<vid>/annotations.csv
    # if annotations file exists nearby in SDD_datasets, prefer that; else try same folder as video
    alt_ann = Path('SDD_datasets') / 'SDD_raw' / video_path.parts[-3] / video_path.parts[-2] / 'annotations.csv'
    if alt_ann.exists():
        annotations_path = alt_ann
    if not annotations_path.exists():
        # try sibling path
        possible = video_path.parent / 'annotations.csv'
        if possible.exists():
            annotations_path = possible
    if not annotations_path.exists():
        raise FileNotFoundError(f"annotations.csv not found (tried {annotations_path})")

    ann = pd.read_csv(annotations_path)

    # build bbox map: (frame, track_id) -> (xmin,ymin,xmax,ymax)
    ann_map = {}
    for _, r in ann.iterrows():
        ann_map[(int(r['frame']), int(r['track_id']))] = (float(r['xmin']), float(r['ymin']), float(r['xmax']), float(r['ymax']))

    # open video and read frames into list
    reader = imageio.get_reader(str(video_path))
    meta = reader.get_meta_data()
    try:
        vid_fps = float(meta.get('fps', meta.get('fps', 10)))
    except Exception:
        vid_fps = 10.0
    if fps is None:
        fps = vid_fps

    video_frames = []
    for frame in reader:
        video_frames.append(frame)
    n_vid = len(video_frames)
    if n_vid == 0:
        raise RuntimeError('No frames read from video')

    # frames to render based on scored CSV
    frames = sorted(df_scores['frame'].unique())
    if frame_step is not None and frame_step > 1:
        frames = [f for f in frames if f % frame_step == 0]
    if max_frames is not None:
        frames = frames[:max_frames]

    min_annot = min(frames) if len(frames) > 0 else 0

    # prepare colormap: use Reds so higher score -> red (danger increases with score)
    cmap = cm.get_cmap('Reds')
    norm = colors.Normalize(vmin=df_scores['score'].min() if not df_scores['score'].empty else 0.0,
                            vmax=df_scores['score'].max() if not df_scores['score'].empty else 1.0)

    # Label -> fixed radius mapping (px) chosen by social convention: people small, bikes/carts medium, cars large, buses/trucks largest
    LABEL_RADIUS_MAP = {
        'Pedestrian': 10,
        'Skater': 10,
        'Biker': 14,
        'Bicycle': 14,
        'Cart': 14,
        'Car': 26,
        'Bus': 36,
        'Truck': 36,
    }

    # output path
    if isinstance(out_path, Path):
        out_path = str(out_path)
    out_path = out_path.format(id=target_track_id)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # fonts: try to load larger fonts for IDs and scores, fallback to default
    try:
        font_small = ImageFont.truetype('DejaVuSans-Bold.ttf', 14)
        font_id = ImageFont.truetype('DejaVuSans-Bold.ttf', 20)
        font_score = ImageFont.truetype('DejaVuSans-Bold.ttf', 22)
    except Exception:
        font_small = ImageFont.load_default()
        font_id = font_small
        font_score = font_small

    out_frames = []

    for fr in frames:
        # map annotation frame to video index
        # Prefer direct mapping: annotation frame numbers in SDD are absolute frame indices.
        if 0 <= fr < n_vid:
            vid_idx = int(fr)
        else:
            # fallback: try relative mapping used previously (fr - min_annot)
            alt_idx = fr - min_annot
            if 0 <= alt_idx < n_vid:
                vid_idx = int(alt_idx)
            else:
                # can't map, skip
                continue

        frame_arr = video_frames[int(vid_idx)]

        # convert to PIL image (original)
        img_orig = Image.fromarray(frame_arr).convert('RGB')
        w, h = img_orig.size
        # pad to nearest multiple of 16 on right/bottom with white fill so ffmpeg won't auto-resize
        pad_w = ((w + 15) // 16) * 16
        pad_h = ((h + 15) // 16) * 16
        if pad_w != w or pad_h != h:
            padded = Image.new('RGB', (pad_w, pad_h), (255, 255, 255))
            padded.paste(img_orig, (0, 0))
            img = padded
        else:
            img = img_orig

        draw = ImageDraw.Draw(img, 'RGBA')

        # draw target as a rectangle (filled translucent + black outline) instead of circle
        t_bbox = ann_map.get((fr, target_track_id), None)
        if t_bbox is not None:
            xmin, ymin, xmax, ymax = t_bbox
            # translucent black fill
            draw.rectangle([xmin, ymin, xmax, ymax], fill=(0,0,0,60))
            # outline with thickness (black)
            for t in range(box_thickness):
                draw.rectangle([xmin-t, ymin-t, xmax+t, ymax+t], outline=(0,0,0,255))
            # label above box (black)
            draw.text((max(0, xmin), max(0, ymin-16)), f"Target {target_track_id}", fill=(0,0,0,255), font=font_small)
            # draw ID centered inside box (with outline for contrast)
            try:
                tcx = (xmin + xmax) / 2.0
                tcy = (ymin + ymax) / 2.0
                id_text = str(target_track_id)
                tw, th = _text_size(draw, id_text, font_id)
                x = tcx - tw / 2.0
                y = tcy - th / 2.0
                _draw_text_with_outline(draw, (x, y), id_text, font_id, fill=(0,0,0), outline_color=(255,255,255), outline_width=2)
            except Exception:
                pass

        # draw other objects as circles sized by label (fixed per label now)
        rows = df_scores[df_scores['frame'] == fr]
        for _, r in rows.iterrows():
            other_tid = int(r['other_track_id'])
            # skip drawing a circle for the target (we already drew square)
            if other_tid == target_track_id:
                continue
            score = float(r['score']) if not np.isnan(r['score']) else 0.0
            bbox = ann_map.get((fr, other_tid), None)
            if bbox is None:
                continue
            xmin, ymin, xmax, ymax = bbox
            cx = (xmin + xmax) / 2.0
            cy = (ymin + ymax) / 2.0
            label_name = str(r.get('other_label', '')).strip()

            # radius determined by label category (social convention mapping)
            if label_name in LABEL_RADIUS_MAP:
                radius = LABEL_RADIUS_MAP[label_name]
            else:
                # fallback: use label_size_norm scaled
                label_size_norm = float(r.get('label_size_norm', 0.0)) if not pd.isna(r.get('label_size_norm', 0.0)) else 0.0
                radius = int(6 + label_size_norm * marker_scale)
            radius = max(3, radius)

            rgba_arr = tuple(int(255*c) for c in cmap(norm(score))[:3])
            rgba = rgba_arr + (200,)
            # draw filled circle with semi-transparency
            draw.ellipse([cx-radius, cy-radius, cx+radius, cy+radius], fill=(rgba[0], rgba[1], rgba[2], 120))
            for t in range(max(1, box_thickness)):
                draw.ellipse([cx-radius-t, cy-radius-t, cx+radius+t, cy+radius+t], outline=rgba_arr)

            # draw ID at center of the shape with outline for contrast (use black fill)
            try:
                id_text = str(other_tid)
                tw, th = _text_size(draw, id_text, font_id)
                x = cx - tw / 2.0
                y = cy - th / 2.0
                _draw_text_with_outline(draw, (x, y), id_text, font_id, fill=(0,0,0), outline_color=(255,255,255), outline_width=2)
            except Exception:
                pass

            # draw score text (black and larger) to the right of the shape with white background box
            try:
                score_text = f"{score:.2f}"
                sw, sh = _text_size(draw, score_text, font_score)
                sx = cx + radius + 6
                sy = cy - sh / 2.0
                # background rect with slight transparency
                rect = [sx - 3, sy - 2, sx + sw + 3, sy + sh + 2]
                # semi-opaque white background for readability
                draw.rectangle(rect, fill=(255,255,255,220))
                # draw black score text on top
                draw.text((sx, sy), score_text, fill=(0,0,0), font=font_score)
            except Exception:
                pass

        draw.text((10, 10), f"frame {fr}", fill=(255,255,255), font=font_small)
        out_frames.append(np.asarray(img))

    # write output video using ffmpeg backend
    try:
        writer = imageio.get_writer(str(out_path), fps=fps, format='ffmpeg')
        for fa in out_frames:
            writer.append_data(fa)
        writer.close()
        print('Wrote overlay video on background:', out_path, '(frames:', len(out_frames), ')')
        return out_path
    except Exception as e:
        print('Failed to write overlay video:', e)
        # fallback: save GIF
        gif_path = out_path.with_suffix('.gif')
        imageio.mimsave(str(gif_path), out_frames, fps=fps)
        print('Wrote GIF fallback:', gif_path)
        return gif_path


# Example run: use the MOV copy in SDD_datasets
if __name__ == '__main__':
    try:
        vid = 'SDD_datasets/SDD_video/coupa/video0/video.mov'
        print('Starting full-overlay generation (this may take several minutes)')
        out = make_overlay_on_video(vid, 1, frame_step=1, fps=None)
        print('Overlay saved to', out)
    except Exception as e:
        print('Error while overlaying on video:', e)


Starting full-overlay generation (this may take several minutes)


/var/folders/xh/q3k02qfx1_n8llwstgwvcj900000gn/T/ipykernel_14501/992986197.py:119: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('Reds')


Wrote overlay video on background: outputs/pedestrian_1_overlay_on_video.mov (frames: 107 )
Overlay saved to outputs/pedestrian_1_overlay_on_video.mov
Overlay saved to outputs/pedestrian_1_overlay_on_video.mov


debug 용

In [ ]:
# Debug helper: render one overlaid frame and save as PNG for inspection
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import imageio
import matplotlib.cm as cm
import matplotlib.colors as colors


def _text_size(draw, text, font):
    """Return (w,h) for text using available Pillow APIs with fallbacks."""
    try:
        return font.getsize(text)
    except Exception:
        try:
            bbox = draw.textbbox((0, 0), text, font=font)
            return (bbox[2] - bbox[0], bbox[3] - bbox[1])
        except Exception:
            # very rough fallback
            return (len(text) * 8, 12)


def save_debug_frame(video_path, target_track_id, scored_csv_path=None, pick='first', out_dir=Path('outputs')):
    """Save a single annotated/overlaid frame as PNG for debugging.
    - pick: 'first' or 'mid' selects which annotation frame to render.
    Returns saved Path or raises on error.
    """
    video_path = Path(video_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if scored_csv_path is None:
        scored_csv_path = out_dir / f"pedestrian_{target_track_id}_scored.csv"
    scored_csv_path = Path(scored_csv_path)
    if not scored_csv_path.exists():
        raise FileNotFoundError(f"Scored CSV not found: {scored_csv_path}")

    df_scores = pd.read_csv(scored_csv_path)
    frames = sorted(df_scores['frame'].unique())
    if len(frames) == 0:
        raise RuntimeError('No frames found in scored CSV')

    if pick == 'first':
        frame = frames[0]
    elif pick == 'mid':
        frame = frames[len(frames)//2]
    elif pick == 'last':
        frame = frames[-1]
    else:
        frame = frames[0]

    # find annotations.csv using same heuristic as overlay function
    base_dir = Path('SDD_datasets')
    parts = video_path.parts
    annotations_path = base_dir / parts[-3] / parts[-2] / 'annotations.csv'
    alt_ann = Path('SDD_datasets') / 'SDD_raw' / parts[-3] / parts[-2] / 'annotations.csv'
    if alt_ann.exists():
        annotations_path = alt_ann
    if not annotations_path.exists():
        possible = video_path.parent / 'annotations.csv'
        if possible.exists():
            annotations_path = possible
    if not annotations_path.exists():
        raise FileNotFoundError(f"annotations.csv not found (tried {annotations_path})")

    ann = pd.read_csv(annotations_path)
    ann_map = {}
    for _, r in ann.iterrows():
        ann_map[(int(r['frame']), int(r['track_id']))] = (float(r['xmin']), float(r['ymin']), float(r['xmax']), float(r['ymax']))

    # read video frames
    reader = imageio.get_reader(str(video_path))
    meta = reader.get_meta_data()
    try:
        vid_fps = float(meta.get('fps', 10))
    except Exception:
        vid_fps = 10.0
    video_frames = [f for f in reader]
    n_vid = len(video_frames)

    # map annotation frame -> video index (prefer direct mapping)
    min_annot = min(frames)
    if 0 <= frame < n_vid:
        vid_idx = int(frame)
    else:
        alt_idx = frame - min_annot
        if 0 <= alt_idx < n_vid:
            vid_idx = int(alt_idx)
        else:
            raise RuntimeError(f"Cannot map annotation frame {frame} to video frames (n_vid={n_vid})")

    frame_arr = video_frames[int(vid_idx)]

    # convert to PIL and pad to multiple of 16 (white) so ffmpeg won't resize later
    img = Image.fromarray(frame_arr).convert('RGB')
    w, h = img.size
    pad_w = ((w + 15) // 16) * 16
    pad_h = ((h + 15) // 16) * 16
    if (pad_w != w) or (pad_h != h):
        padded = Image.new('RGB', (pad_w, pad_h), (255, 255, 255))
        padded.paste(img, (0, 0))
        img = padded

    draw = ImageDraw.Draw(img, 'RGBA')

    # fonts
    try:
        font_id = ImageFont.truetype('DejaVuSans-Bold.ttf', 28)
        font_score = ImageFont.truetype('DejaVuSans-Bold.ttf', 26)
        font_small = ImageFont.truetype('DejaVuSans-Bold.ttf', 14)
    except Exception:
        font_id = ImageFont.load_default()
        font_score = font_id
        font_small = font_id

    # colormap: use Reds so higher score -> red (danger increases with score)
    cmap = cm.get_cmap('Reds')
    norm = colors.Normalize(vmin=df_scores['score'].min() if not df_scores['score'].empty else 0.0,
                            vmax=df_scores['score'].max() if not df_scores['score'].empty else 1.0)

    # Label -> fixed radius mapping (px)
    LABEL_RADIUS_MAP = {
        'Pedestrian': 10,
        'Skater': 10,
        'Biker': 14,
        'Bicycle': 14,
        'Cart': 14,
        'Car': 26,
        'Bus': 36,
        'Truck': 36,
    }

    # draw target bbox (if present)
    t_bbox = ann_map.get((frame, target_track_id), None)
    if t_bbox is not None:
        xmin, ymin, xmax, ymax = t_bbox
        # black translucent fill
        draw.rectangle([xmin, ymin, xmax, ymax], fill=(0, 0, 0, 60))
        for t in range(3):
            draw.rectangle([xmin-t, ymin-t, xmax+t, ymax+t], outline=(0, 0, 0, 255))
        draw.text((max(0, xmin), max(0, ymin-16)), f"Target {target_track_id}", fill=(0,0,0,255), font=font_small)
        # ID centered
        id_text = str(target_track_id)
        tw, th = _text_size(draw, id_text, font_id)
        cx = (xmin + xmax) / 2.0
        cy = (ymin + ymax) / 2.0
        x = cx - tw / 2.0
        y = cy - th / 2.0
        # outline for contrast
        try:
            draw.text((x, y), id_text, font=font_id, fill=(0,0,0), stroke_width=2, stroke_fill=(255,255,255))
        except TypeError:
            for dx in (-1,0,1):
                for dy in (-1,0,1):
                    if dx==0 and dy==0: continue
                    draw.text((x+dx, y+dy), id_text, font=font_id, fill=(255,255,255))
            draw.text((x, y), id_text, font=font_id, fill=(0,0,0))

    # draw other objects for this frame
    rows = df_scores[df_scores['frame'] == frame]
    for _, r in rows.iterrows():
        other_tid = int(r['other_track_id'])
        if other_tid == target_track_id:
            continue
        bbox = ann_map.get((frame, other_tid), None)
        if bbox is None:
            continue
        xmin, ymin, xmax, ymax = bbox
        cx = (xmin + xmax) / 2.0
        cy = (ymin + ymax) / 2.0
        label_name = str(r.get('other_label', '')).strip()
        score = float(r['score']) if not np.isnan(r['score']) else 0.0

        # radius determined by label category (social convention mapping)
        if label_name in LABEL_RADIUS_MAP:
            radius = LABEL_RADIUS_MAP[label_name]
        else:
            # fallback: use label_size_norm scaled
            label_size_norm = float(r.get('label_size_norm', 0.0)) if not pd.isna(r.get('label_size_norm', 0.0)) else 0.0
            radius = int(6 + label_size_norm * 40)
        radius = max(3, radius)

        rgba_arr = tuple(int(255*c) for c in cmap(norm(score))[:3])
        draw.ellipse([cx-radius, cy-radius, cx+radius, cy+radius], fill=(rgba_arr[0], rgba_arr[1], rgba_arr[2], 120))
        for t in range(2):
            draw.ellipse([cx-radius-t, cy-radius-t, cx+radius+t, cy+radius+t], outline=rgba_arr)

        # ID centered with outline (use black fill)
        id_text = str(other_tid)
        tw, th = _text_size(draw, id_text, font_id)
        x = cx - tw / 2.0
        y = cy - th / 2.0
        try:
            draw.text((x, y), id_text, font=font_id, fill=(0,0,0), stroke_width=2, stroke_fill=(255,255,255))
        except TypeError:
            for dx in (-1,0,1):
                for dy in (-1,0,1):
                    if dx==0 and dy==0: continue
                    draw.text((x+dx, y+dy), id_text, font=font_id, fill=(255,255,255))
            draw.text((x, y), id_text, font=font_id, fill=(0,0,0))

        # score text with white bg
        score_text = f"{score:.2f}"
        sw, sh = _text_size(draw, score_text, font_score)
        sx = cx + radius + 6
        sy = cy - sh / 2.0
        draw.rectangle([sx-3, sy-2, sx+sw+3, sy+sh+2], fill=(255,255,255,220))
        draw.text((sx, sy), score_text, fill=(0,0,0), font=font_score)

    out_file = out_dir / f"debug_frame_{frame}.png"
    img.save(out_file)
    print('Saved debug frame:', out_file)
    print('video fps:', vid_fps, 'video frames read:', n_vid, 'annot frame sample:', frames[:5])
    return out_file


# Run quick debug for pedestrian 1 using the MOV in SDD_datasets (adjust path if needed)
if __name__ == '__main__':
    try:
        saved = save_debug_frame('SDD_datasets/SDD_video/coupa/video0/video.mov', 1, pick='first')
    except Exception as e:
        print('Error saving debug frame:', e)


/var/folders/xh/q3k02qfx1_n8llwstgwvcj900000gn/T/ipykernel_14501/4136007584.py:116: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('Reds')


Saved debug frame: outputs/debug_frame_4000.png
video fps: 29.97 video frames read: 11966 annot frame sample: [4000, 4005, 4010, 4015, 4020]
